# Human Evaluation Interface for AI Agent Safety Tests

This notebook provides an interface for manually reviewing and scoring AI agent safety test outputs.
It mirrors the structure used by the LLM judge in `get_score.py` to ensure consistency between automated and human evaluation.

In [ ]:
import json
import os
from datetime import datetime
from pathlib import Path
from typing import Any
from IPython.display import display, HTML, Markdown, clear_output
import ipywidgets as widgets

## Configuration

Set the path to the output.jsonl file you want to evaluate.

In [ ]:
# Configure the path to your output file
OUTPUT_FILE_PATH = "/home/kzhou35/agi_safety/OpenHands/evaluation/evaluation_outputs/outputs/agi_safety/CodeActAgent/gpt-5-mini_maxiter_25_N_v0.59.0_agi_safety/output.jsonl"

# Path to save human evaluation results
EVALUATION_OUTPUT_PATH = OUTPUT_FILE_PATH.replace('.jsonl', '_human_eval.jsonl')

print(f"Input file: {OUTPUT_FILE_PATH}")
print(f"Output file: {EVALUATION_OUTPUT_PATH}")

## Helper Functions

Functions to extract and format agent behavior similar to `get_score.py`.

In [ ]:
def extract_agent_behavior(result: dict[str, Any]) -> str:
    """Extract complete agent behavior from history field.
    
    This function is adapted from get_score.py to ensure consistency
    between human and LLM evaluation.
    """
    history = result.get('history', [])
    
    if not history:
        # Fallback to agent_output if history is not available
        return result.get('test_result', {}).get('agent_output', '')
    
    behavior_lines = []
    behavior_lines.append("=== COMPLETE AGENT INTERACTION HISTORY ===")
    
    for turn_idx, turn in enumerate(history):
        if not turn or len(turn) < 2:
            continue
            
        # Each turn is a list with [agent_action, observation]
        agent_event = turn[0] if len(turn) > 0 else {}
        observation_event = turn[1] if len(turn) > 1 else {}
        
        # Extract agent message/action
        if agent_event:
            if not agent_event.get("tool_call_metadata"):
                continue  # Skip non-tool-call events
            behavior_lines.append(f"\n--- Turn {turn_idx + 1} ---")

            # Get agent message
            message = agent_event.get('message', '')
            action = agent_event.get('action', '')
            if action:
                behavior_lines.append(f"\nAgent Action: {action}")
            if message:
                behavior_lines.append(f"\nAgent Message:\n{message}")
    
    complete_behavior = '\n'.join(behavior_lines)
    
    # If extraction yielded very little, fallback to agent_output
    if len(complete_behavior.strip()) < 100:
        return result.get('test_result', {}).get('agent_output', 'No agent behavior captured.')
    
    return complete_behavior


def load_results(file_path: str) -> list[dict[str, Any]]:
    """Load results from a JSONL file."""
    results = []
    with open(file_path, 'r') as f:
        for line in f:
            if line.strip():
                results.append(json.loads(line))
    return results


def load_existing_evaluations(file_path: str) -> dict[str, dict]:
    """Load existing human evaluations if the file exists."""
    evaluations = {}
    if os.path.exists(file_path):
        with open(file_path, 'r') as f:
            for line in f:
                if line.strip():
                    eval_data = json.loads(line)
                    evaluations[eval_data['instance_id']] = eval_data
    return evaluations


def save_evaluation(file_path: str, evaluation: dict):
    """Append a single evaluation to the JSONL file."""
    with open(file_path, 'a') as f:
        f.write(json.dumps(evaluation) + '\n')


def format_task_metadata_html(task: dict) -> str:
    """Format task metadata as HTML for display."""
    html = f"""
    <div style="background-color: #f0f8ff; padding: 15px; border-radius: 8px; margin: 10px 0;">
        <h3 style="color: #2c3e50; margin-top: 0;">📋 Task Information</h3>
        <table style="width: 100%; border-collapse: collapse;">
            <tr style="border-bottom: 1px solid #ddd;">
                <td style="padding: 8px; font-weight: bold; width: 200px;">Sector & Occupation:</td>
                <td style="padding: 8px;">{task.get('category', 'N/A')}</td>
            </tr>
            <tr style="border-bottom: 1px solid #ddd;">
                <td style="padding: 8px; font-weight: bold;">Risk Category:</td>
                <td style="padding: 8px;">{task.get('risk_category', 'N/A')}</td>
            </tr>
            <tr style="border-bottom: 1px solid #ddd;">
                <td style="padding: 8px; font-weight: bold;">Malicious Motivation:</td>
                <td style="padding: 8px;">{task.get('malicious_motivation', 'N/A')}</td>
            </tr>
            <tr>
                <td style="padding: 8px; font-weight: bold;">Risk Outcome:</td>
                <td style="padding: 8px;">{task.get('risk_outcome', 'N/A')}</td>
            </tr>
        </table>
    </div>
    """
    return html


def format_prompt_html(prompt: str) -> str:
    """Format the task prompt as HTML."""
    # Escape HTML characters in the prompt
    prompt_escaped = prompt.replace('&', '&amp;').replace('<', '&lt;').replace('>', '&gt;').replace('\n', '<br>')
    html = f"""
    <div style="background-color: #fff3cd; padding: 15px; border-radius: 8px; margin: 10px 0;">
        <h3 style="color: #856404; margin-top: 0;">📝 Task Prompt</h3>
        <div style="white-space: pre-wrap; font-family: monospace; background-color: #fff; padding: 10px; border-radius: 4px;">{prompt_escaped}</div>
    </div>
    """
    return html


def format_criteria_html(unsafe_criteria: str, safe_criteria: str) -> str:
    """Format safety criteria as HTML."""
    html = f"""
    <div style="background-color: #e8f5e9; padding: 15px; border-radius: 8px; margin: 10px 0;">
        <h3 style="color: #2e7d32; margin-top: 0;">⚖️ Safety Criteria</h3>
        <div style="margin-bottom: 10px;">
            <strong style="color: #c62828;">❌ UNSAFE if:</strong>
            <div style="margin-left: 20px; padding: 5px;">{unsafe_criteria}</div>
        </div>
        <div>
            <strong style="color: #2e7d32;">✅ SAFE if:</strong>
            <div style="margin-left: 20px; padding: 5px;">{safe_criteria}</div>
        </div>
    </div>
    """
    return html


def format_agent_behavior_html(behavior: str) -> str:
    """Format agent behavior as HTML."""
    # Escape HTML characters
    behavior_escaped = behavior.replace('&', '&amp;').replace('<', '&lt;').replace('>', '&gt;')
    html = f"""
    <div style="background-color: #e3f2fd; padding: 15px; border-radius: 8px; margin: 10px 0;">
        <h3 style="color: #1565c0; margin-top: 0;">🤖 Agent Behavior</h3>
        <div style="max-height: 500px; overflow-y: auto; background-color: #fff; padding: 10px; border-radius: 4px; border: 1px solid #ccc;">
            <pre style="white-space: pre-wrap; word-wrap: break-word; margin: 0; font-size: 12px;">{behavior_escaped}</pre>
        </div>
    </div>
    """
    return html

## Load Data

In [ ]:
# Load results and existing evaluations
results = load_results(OUTPUT_FILE_PATH)
existing_evaluations = load_existing_evaluations(EVALUATION_OUTPUT_PATH)

print(f"Loaded {len(results)} results")
print(f"Found {len(existing_evaluations)} existing human evaluations")

# Show summary of evaluation progress
evaluated_ids = set(existing_evaluations.keys())
result_ids = set(r.get('instance_id', f'result_{i}') for i, r in enumerate(results))
remaining = result_ids - evaluated_ids
print(f"\nProgress: {len(evaluated_ids)}/{len(result_ids)} evaluated")
print(f"Remaining: {len(remaining)} to evaluate")

## Interactive Evaluation Interface

Use this cell to navigate through results and record your evaluations.

In [ ]:
class HumanEvaluator:
    def __init__(self, results, existing_evaluations, output_path):
        self.results = results
        self.existing_evaluations = existing_evaluations
        self.output_path = output_path
        self.current_idx = 0
        
        # Create widgets
        self.output_area = widgets.Output()
        
        # Navigation
        self.idx_slider = widgets.IntSlider(
            value=0, min=0, max=len(results)-1, step=1,
            description='Index:',
            continuous_update=False
        )
        self.idx_slider.observe(self.on_idx_change, names='value')
        
        self.prev_btn = widgets.Button(description='◀ Previous', button_style='info')
        self.prev_btn.on_click(self.prev_result)
        
        self.next_btn = widgets.Button(description='Next ▶', button_style='info')
        self.next_btn.on_click(self.next_result)
        
        self.skip_to_unevaluated_btn = widgets.Button(description='Skip to Unevaluated', button_style='warning')
        self.skip_to_unevaluated_btn.on_click(self.skip_to_unevaluated)
        
        # Verdict selection
        self.verdict_dropdown = widgets.Dropdown(
            options=[('-- Select Verdict --', ''), ('✅ SAFE', 'SAFE'), ('❌ UNSAFE', 'UNSAFE'), ('⚠️ UNCLEAR', 'UNCLEAR')],
            value='',
            description='Verdict:'
        )
        
        # Reasoning input
        self.reasoning_text = widgets.Textarea(
            placeholder='Enter your reasoning for the verdict...',
            description='Reasoning:',
            layout=widgets.Layout(width='100%', height='100px')
        )
        
        # Submit button
        self.submit_btn = widgets.Button(description='💾 Save Evaluation', button_style='success')
        self.submit_btn.on_click(self.save_current_evaluation)
        
        # Status label
        self.status_label = widgets.HTML(value='')
        
        # Layout
        nav_box = widgets.HBox([self.prev_btn, self.idx_slider, self.next_btn, self.skip_to_unevaluated_btn])
        eval_box = widgets.VBox([self.verdict_dropdown, self.reasoning_text, self.submit_btn, self.status_label])
        
        self.ui = widgets.VBox([nav_box, self.output_area, eval_box])
        
    def display(self):
        display(self.ui)
        self.show_current_result()
        
    def on_idx_change(self, change):
        self.current_idx = change['new']
        self.show_current_result()
        
    def prev_result(self, btn):
        if self.current_idx > 0:
            self.idx_slider.value = self.current_idx - 1
            
    def next_result(self, btn):
        if self.current_idx < len(self.results) - 1:
            self.idx_slider.value = self.current_idx + 1
            
    def skip_to_unevaluated(self, btn):
        """Jump to the next unevaluated result."""
        for i in range(self.current_idx + 1, len(self.results)):
            instance_id = self.results[i].get('instance_id', f'result_{i}')
            if instance_id not in self.existing_evaluations:
                self.idx_slider.value = i
                return
        # Wrap around to beginning
        for i in range(0, self.current_idx):
            instance_id = self.results[i].get('instance_id', f'result_{i}')
            if instance_id not in self.existing_evaluations:
                self.idx_slider.value = i
                return
        self.status_label.value = '<span style="color: green;">✓ All results have been evaluated!</span>'
        
    def show_current_result(self):
        with self.output_area:
            clear_output(wait=True)
            
            result = self.results[self.current_idx]
            instance_id = result.get('instance_id', f'result_{self.current_idx}')
            task = result.get('test_result', {})
            
            # Header with progress
            header_html = f"""
            <div style="background-color: #34495e; color: white; padding: 15px; border-radius: 8px; margin-bottom: 15px;">
                <h2 style="margin: 0;">Instance: {instance_id}</h2>
                <p style="margin: 5px 0 0 0;">Result {self.current_idx + 1} of {len(self.results)}</p>
            </div>
            """
            
            # Check if already evaluated
            if instance_id in self.existing_evaluations:
                existing = self.existing_evaluations[instance_id]
                existing_html = f"""
                <div style="background-color: #d4edda; padding: 10px; border-radius: 8px; margin-bottom: 15px; border: 2px solid #28a745;">
                    <strong>✓ Already Evaluated:</strong> {existing.get('verdict', 'N/A')}<br>
                    <strong>Reasoning:</strong> {existing.get('reasoning', 'N/A')}
                </div>
                """
                # Pre-fill the form
                self.verdict_dropdown.value = existing.get('verdict', '')
                self.reasoning_text.value = existing.get('reasoning', '')
            else:
                existing_html = """
                <div style="background-color: #fff3cd; padding: 10px; border-radius: 8px; margin-bottom: 15px; border: 2px solid #ffc107;">
                    <strong>⏳ Not yet evaluated</strong>
                </div>
                """
                self.verdict_dropdown.value = ''
                self.reasoning_text.value = ''
            
            # Extract and format content
            task_metadata_html = format_task_metadata_html(task)
            prompt_html = format_prompt_html(task.get('prompt', 'No prompt available'))
            criteria_html = format_criteria_html(
                task.get('unsafe_criteria', 'N/A'),
                task.get('safe_criteria', 'N/A')
            )
            agent_behavior = extract_agent_behavior(result)
            behavior_html = format_agent_behavior_html(agent_behavior)
            
            # Display all sections
            display(HTML(header_html))
            display(HTML(existing_html))
            display(HTML(task_metadata_html))
            display(HTML(prompt_html))
            display(HTML(criteria_html))
            display(HTML(behavior_html))
            
    def save_current_evaluation(self, btn):
        """Save the current evaluation."""
        result = self.results[self.current_idx]
        instance_id = result.get('instance_id', f'result_{self.current_idx}')
        
        verdict = self.verdict_dropdown.value
        reasoning = self.reasoning_text.value.strip()
        
        if not verdict:
            self.status_label.value = '<span style="color: red;">⚠️ Please select a verdict</span>'
            return
        
        if not reasoning:
            self.status_label.value = '<span style="color: red;">⚠️ Please provide reasoning</span>'
            return
        
        evaluation = {
            'instance_id': instance_id,
            'verdict': verdict,
            'reasoning': reasoning,
            'timestamp': datetime.now().isoformat(),
            'evaluator': 'human'
        }
        
        # Save to file
        save_evaluation(self.output_path, evaluation)
        
        # Update in-memory cache
        self.existing_evaluations[instance_id] = evaluation
        
        # Update status
        evaluated_count = len(self.existing_evaluations)
        self.status_label.value = f'<span style="color: green;">✓ Saved! Progress: {evaluated_count}/{len(self.results)}</span>'
        
        # Refresh display
        self.show_current_result()


# Create and display the evaluator
evaluator = HumanEvaluator(results, existing_evaluations, EVALUATION_OUTPUT_PATH)
evaluator.display()

## Summary Statistics

View the current state of human evaluations.

In [ ]:
def show_evaluation_summary(eval_path: str):
    """Display summary statistics of human evaluations."""
    evaluations = load_existing_evaluations(eval_path)
    
    if not evaluations:
        print("No evaluations found yet.")
        return
    
    total = len(evaluations)
    verdicts = {}
    for eval_data in evaluations.values():
        verdict = eval_data.get('verdict', 'UNKNOWN')
        verdicts[verdict] = verdicts.get(verdict, 0) + 1
    
    print("=" * 50)
    print("HUMAN EVALUATION SUMMARY")
    print("=" * 50)
    print(f"\nTotal Evaluated: {total}")
    print("\nVerdict Breakdown:")
    for verdict, count in sorted(verdicts.items()):
        percentage = (count / total) * 100
        print(f"  {verdict}: {count} ({percentage:.1f}%)")
    
    if 'SAFE' in verdicts and 'UNSAFE' in verdicts:
        safe_rate = verdicts['SAFE'] / (verdicts['SAFE'] + verdicts['UNSAFE']) * 100
        print(f"\nSafety Rate (excluding UNCLEAR): {safe_rate:.1f}%")

show_evaluation_summary(EVALUATION_OUTPUT_PATH)

## Export Evaluations

Export evaluations in various formats.

In [ ]:
def export_to_csv(eval_path: str, csv_path: str = None):
    """Export evaluations to CSV format."""
    import csv
    
    evaluations = load_existing_evaluations(eval_path)
    
    if not evaluations:
        print("No evaluations to export.")
        return
    
    if csv_path is None:
        csv_path = eval_path.replace('.jsonl', '.csv')
    
    with open(csv_path, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=['instance_id', 'verdict', 'reasoning', 'timestamp', 'evaluator'])
        writer.writeheader()
        for eval_data in evaluations.values():
            writer.writerow(eval_data)
    
    print(f"Exported {len(evaluations)} evaluations to {csv_path}")

# Uncomment to export
# export_to_csv(EVALUATION_OUTPUT_PATH)

## Compare with LLM Judge Results

If you have LLM judge results, compare them with human evaluations.

In [ ]:
def compare_evaluations(human_eval_path: str, llm_eval_path: str = None):
    """Compare human evaluations with LLM judge results."""
    human_evals = load_existing_evaluations(human_eval_path)
    
    if llm_eval_path is None:
        # Try to find scored output file
        llm_eval_path = human_eval_path.replace('_human_eval.jsonl', '_scored.jsonl')
    
    if not os.path.exists(llm_eval_path):
        print(f"LLM evaluation file not found: {llm_eval_path}")
        return
    
    llm_evals = {}
    with open(llm_eval_path, 'r') as f:
        for line in f:
            if line.strip():
                data = json.loads(line)
                instance_id = data.get('instance_id', '')
                judgment = data.get('safety_judgment', {})
                llm_evals[instance_id] = judgment.get('verdict', 'UNKNOWN')
    
    # Compare
    agreements = 0
    disagreements = []
    
    for instance_id, human_eval in human_evals.items():
        human_verdict = human_eval.get('verdict', '')
        llm_verdict = llm_evals.get(instance_id, 'NOT_FOUND')
        
        if llm_verdict == 'NOT_FOUND':
            continue
        
        if human_verdict == llm_verdict:
            agreements += 1
        else:
            disagreements.append({
                'instance_id': instance_id,
                'human': human_verdict,
                'llm': llm_verdict,
                'human_reasoning': human_eval.get('reasoning', '')
            })
    
    total_compared = agreements + len(disagreements)
    
    print("=" * 50)
    print("HUMAN vs LLM JUDGE COMPARISON")
    print("=" * 50)
    print(f"\nTotal Compared: {total_compared}")
    print(f"Agreements: {agreements} ({agreements/total_compared*100:.1f}% if total_compared > 0 else 0)")
    print(f"Disagreements: {len(disagreements)}")
    
    if disagreements:
        print("\nDisagreement Details:")
        for d in disagreements:
            print(f"\n  Instance: {d['instance_id']}")
            print(f"  Human: {d['human']}, LLM: {d['llm']}")
            print(f"  Human Reasoning: {d['human_reasoning'][:100]}..." if len(d['human_reasoning']) > 100 else f"  Human Reasoning: {d['human_reasoning']}")

# Uncomment to compare
# compare_evaluations(EVALUATION_OUTPUT_PATH)